# Movie rating classification

## main purpose
Our main purpose  here given a set of movies and set of users rating the movies we want find model that
predict given a movie and user what the user will rate this movie

## data set
first we will install the data set

In [105]:
from google.colab import files
files.upload()

Saving cosine_similarity.py to cosine_similarity (2).py
Saving main.py to main (3).py
Saving matrix_factorization.py to matrix_factorization (3).py
Saving out to out (3)
Saving README.md to README (3).md
Saving resarch.ipynb to resarch (3).ipynb
Saving run to run (3)
Saving SGD_matrix_factorization.py to SGD_matrix_factorization (3).py
Saving two_tower.py to two_tower (2).py
Saving two_tower_model.py to two_tower_model (4).py


{'cosine_similarity (2).py': b'from sklearn.base import BaseEstimator, RegressorMixin\nfrom sklearn.metrics.pairwise import cosine_similarity\n\n\nclass ContentBasedModel(BaseEstimator, RegressorMixin):\n\n    def __init__(self, user_features, movie_features):\n        self.user_features = user_features\n        self.movie_features = movie_features\n\n    def fit(self, X, y):\n        return self\n\n    def predict(self, X):\n\n        user_id = X["userId"].iloc[0]\n\n        user_vector = self.user_features.loc[[user_id]]\n\n        scores = cosine_similarity(\n            user_vector,\n            self.movie_features\n        ).ravel()\n\n        return scores',
 'main (3).py': b'from two_tower_model import TwoTowerModel\nimport time \nimport pandas as pd\n\ndef build_movies():\n    pd.\n\ndef main():',
 'matrix_factorization (3).py': b'from sklearn.decomposition import NMF\nfrom scipy.sparse import csr_matrix\nimport numpy as np\n\n\nclass SparseNMF(NMF):\n\n    def __init__(\n     

In [106]:
%pip install kagglehub
import kagglehub

# Download latest version
path = kagglehub.dataset_download("grouplens/movielens-20m-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'movielens-20m-dataset' dataset.
Path to dataset files: /kaggle/input/movielens-20m-dataset


In [107]:
%pip install tensorflow
%pip install matplot
from matplotlib import pylab
#from google.colab import drive


import matplotlib.pyplot as plt
%pip install pandas
import pandas as pd
import sys
!{sys.executable} -m pip install scikit-learn
!{sys.executable} -m pip install seaborn
from sklearn.model_selection import train_test_split
import numpy as np
import seaborn as sns

### this are the files we get from kaggle:

In [108]:
import os
print(os.listdir(path))

['rating.csv', 'link.csv', 'genome_tags.csv', 'genome_scores.csv', 'tag.csv', 'movie.csv']


### rating

In [109]:
ratings = pd.read_csv(os.path.join(path, 'rating.csv'), nrows=1_000_000
)
ratings = ratings.drop(columns=['timestamp'])
ratings.head(5)

,userId,movieId,rating
0,1,2,3.5
1,1,29,3.5
2,1,32,3.5
3,1,47,3.5
4,1,50,3.5


### movies

In [110]:
movies = pd.read_csv(os.path.join(path, 'movie.csv'))
movies.head(5)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## Normalizations

### Encode genres

In [111]:
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()
if "genres" in movies.columns:
    genre_features = mlb.fit_transform(movies['genres'].str.split('|'))
    genre_df = pd.DataFrame(
        genre_features,
        columns=mlb.classes_,
        index=movies.index
    )

    movies = pd.concat(
        [movies[["movieId"]], genre_df],
        axis=1
    )
movies.head(5)

,movieId,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,0,0,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,0,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,0,0,0,0,0,1,0,0,1,...,0,0,0,0,0,1,0,0,0,0
4,5,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Learing

now we will suggest a learning algorithm for this problem

In [112]:
X_train, X_test, y_train, y_test = train_test_split(
    ratings[["userId", "movieId"]],
    ratings["rating"],
    test_size=0.2,
    random_state=42
)


## LinerRegression

In [113]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)


LinearRegression()

### score

In [114]:
def print_error(model, X_train, y_train, X_test, y_test):
    from sklearn.metrics import mean_absolute_error, mean_squared_error

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_mae = mean_absolute_error(y_train, train_pred)
    test_mae = mean_absolute_error(y_test, test_pred)

    train_rmse = mean_squared_error(y_train, train_pred) ** 0.5
    test_rmse = mean_squared_error(y_test, test_pred) ** 0.5

    print("Train MAE:", train_mae)
    print("Test MAE:", test_mae)
    print("Train RMSE:", train_rmse)
    print("Test RMSE:", test_rmse)

print_error(model, X_train, y_train, X_test, y_test)

Train MAE: 0.8405564838565901
Test MAE: 0.8396936110796265
Train RMSE: 1.0522522023096548
Test RMSE: 1.051267197872636


As we can see we get a bad RMSE so we need to choose another model class

## Matrix Factorization


In [115]:

from matrix_factorization import SparseNMF

model = SparseNMF(movies.shape[0])
model.fit(X_train,y_train)

SparseNMF(num_of_movies=27278)

## Testing

In [116]:
from sklearn.metrics import mean_squared_error

def print_RMSE(model,X_train,X_test,y_train,y_test):
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_mse = np.sqrt(mean_squared_error(y_train, train_pred))
    test_mse = np.sqrt(mean_squared_error(y_test, test_pred))

    print("Train RMSE:", train_mse)
    print("Test RMSE:", test_mse)

print_RMSE(model,X_train,X_test,y_train,y_test)

Train RMSE: 2.6516475508306607
Test RMSE: 2.7035846623622786


As we can see this soultion doesn't work well against LinearRegression. In addition this model is not that scaleable because we use a constant matrix.

## SGD Matrix Factorization

In [117]:
!pip install scikit-surprise

In [118]:
from SGD_matrix_factorization import SGDMatrixFacorization

model = SGDMatrixFacorization(rating_scale=(0.0,5.0))
model.fit(X_train,y_train)

SGDMatrixFacorization(rating_scale=(0.0, 5.0))

## testing

In [119]:
known_movies = set(model.model.trainset._raw2inner_id_items.keys())

mask = X_test["movieId"].isin(known_movies)

X_test_filtered = X_test[mask]
y_test_filtered = y_test[mask]

print_RMSE(model=model,X_train=X_train,X_test=X_test_filtered,y_train=y_train,y_test=y_test_filtered)

Train RMSE: 0.5366949138863938
Test RMSE: 0.8468428807052385


As we can see, the model achieves good performance. However, it requires all users and movies to be known during training. Therefore, it cannot naturally make predictions for previously unseen movies or users (the cold-start problem).

## Two Tower

we need to repreasent users base on what they like

In [120]:
gerners = movies.drop(columns=["movieId"])
genre_counts = gerners.sum()
genre_counts

,0
(no genres listed),246
Action,3520
Adventure,2329
Animation,1027
Children,1139
Comedy,8374
Crime,2939
Documentary,2471
Drama,13344
Fantasy,1412


In [121]:
user_genre = ratings.merge(movies, on="movieId")
genre_columns = gerners.columns.to_list()

user_genre_matrix = pd.DataFrame(index=user_genre["userId"].unique())

for genre in genre_columns:
    user_genre_matrix[genre] = (
        user_genre[user_genre[genre] == 1]
        .groupby("userId")["rating"]
        .mean()
    )
genre_means = user_genre_matrix.mean()
user_genre_matrix = user_genre_matrix.fillna(genre_means)
user_genre_matrix

,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
1,3.333333,3.727273,3.787671,3.650000,3.605263,3.731707,3.809524,3.773979,3.767442,3.789855,3.900266,3.744444,4.250000,3.666667,3.611111,3.954545,3.712500,3.761905,3.666667,3.375000
2,3.333333,4.631579,4.823529,3.000000,3.000000,3.900000,5.000000,3.773979,3.894737,2.000000,5.000000,3.555556,3.000000,3.000000,4.500000,3.833333,4.608696,4.263158,4.250000,4.500000
3,3.333333,4.114754,4.220000,3.750000,4.300000,4.057692,4.285714,4.000000,4.224138,4.300000,3.000000,3.937500,3.734584,4.000000,4.363636,4.062500,4.000000,4.260000,4.666667,4.333333
4,3.333333,3.538462,3.833333,4.000000,3.750000,3.545455,3.166667,3.773979,3.750000,3.666667,3.900266,3.416261,3.734584,4.000000,2.666667,3.500000,3.000000,3.461538,4.000000,4.000000
5,3.333333,4.500000,4.523810,4.666667,4.181818,4.083333,4.142857,3.773979,4.185185,3.727273,3.900266,3.000000,5.000000,4.375000,3.500000,3.937500,4.600000,4.333333,4.000000,5.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6739,3.333333,2.135514,1.989950,1.761905,1.595745,2.455385,3.067164,3.200000,2.970899,2.173469,4.050000,2.118110,2.000000,1.918367,3.132353,2.569343,2.167785,2.635628,3.222222,2.388889
6740,3.333333,4.000000,4.000000,5.000000,5.000000,3.650000,4.166667,3.773979,3.772727,3.875000,3.900266,3.416261,3.734584,3.549957,4.000000,3.928571,3.000000,3.750000,3.875000,4.500000
6741,3.333333,3.250000,3.660000,4.090909,3.900000,4.103448,4.083333,4.000000,3.865385,3.725000,3.000000,4.500000,3.600000,3.687500,3.900000,3.937500,3.800000,4.045455,4.000000,3.570488
6742,3.333333,3.500000,4.000000,3.615842,3.453785,4.500000,4.000000,3.773979,4.625000,5.000000,3.900266,3.416261,5.000000,3.549957,3.719430,4.200000,3.466998,4.000000,4.666667,3.000000


In [122]:
if "movieId" in movies.columns:
    movies = movies.set_index("movieId")


In [123]:
from two_tower_model import TwoTowerModel
embedding_dim = 32
batch_size = 4*1024
epochs = 5
model = TwoTowerModel(user_features=user_genre_matrix,movie_features=movies,
                      epochs=epochs,batch_size=batch_size,embedding_dim=embedding_dim)
model.fit(X_train,y_train)

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - loss: 1.1522 - root_mean_squared_error: 1.0734 - val_loss: 0.8539 - val_root_mean_squared_error: 0.9241
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.8385 - root_mean_squared_error: 0.9157 - val_loss: 0.8281 - val_root_mean_squared_error: 0.9100
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.8160 - root_mean_squared_error: 0.9034 - val_loss: 0.8068 - val_root_mean_squared_error: 0.8982
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.8029 - root_mean_squared_error: 0.8960 - val_loss: 0.7972 - val_root_mean_squared_error: 0.8929
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.7943 - root_mean_squared_error: 0.8912 - val_loss: 0.7884 - val_root_mean_squared_error: 0.8879


TwoTowerModel(batch_size=4096, embedding_dim=32, epochs=5,
              movie_features=array([[0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 1., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.]], dtype=float32),
              user_features=array([[3.3333333, 3.7272727, 3.7876713, ..., 3.7619047, 3.6666667,
        3.375    ],
       [3.3333333, 4.631579 , 4.8235292, ..., 4.263158 , 4.25     ,
        4.5      ],
       [3.3333333, 4.114754 , 4.22     , ..., 4.26     , 4.6666665,
        4.3333335],
       ...,
       [3.3333333, 3.25     , 3.66     , ..., 4.0454545, 4.       ,
        3.5704882],
       [3.3333333, 3.5      , 4.       , ..., 4.       , 4.6666665,
        3.       ],
       [3.3333333, 3.3333333, 3.3113208, ..., 3.440678 , 3.7857144,
        3.4166667]], dtype=float32))

## Testing

In [124]:
print_RMSE(model,X_train,X_test,y_train,y_test)

Train RMSE: 0.887891134850858
Test RMSE: 0.8855830758908975


## Tunning
lets do tunning for the model

In [125]:
from two_tower_model import TwoTowerModel
learning_rates = [
    1e-4,
    3e-4,
    1e-3,
    3e-3,
    1e-2
]

embedding_dim = 32
batch_size = 4*1024
epochs = 7
model = TwoTowerModel(user_features=user_genre_matrix,movie_features=movies,
                      epochs=epochs,batch_size=batch_size,embedding_dim=embedding_dim,
                      learning_rate= learning_rates[3])
model.fit(X_train,y_train)

Epoch 1/7
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 1.0794 - root_mean_squared_error: 1.0390 - val_loss: 0.8363 - val_root_mean_squared_error: 0.9145
Epoch 2/7
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.8237 - root_mean_squared_error: 0.9076 - val_loss: 0.8153 - val_root_mean_squared_error: 0.9029
Epoch 3/7
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.8033 - root_mean_squared_error: 0.8963 - val_loss: 0.7983 - val_root_mean_squared_error: 0.8935
Epoch 4/7
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.7928 - root_mean_squared_error: 0.8904 - val_loss: 0.7787 - val_root_mean_squared_error: 0.8824
Epoch 5/7
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.7855 - root_mean_squared_error: 0.8863 - val_loss: 0.7761 - val_root_mean_squared_error: 0.8810
Epoch 6/7
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.7757 - root_mean_squared_error: 0.8807 - val_loss: 0.7777 - val_root_mean_squared_error: 0.8819
Epoch 7/7
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss

TwoTowerModel(batch_size=4096, embedding_dim=32, epochs=7, learning_rate=0.003,
              movie_features=array([[0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 1., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.]], dtype=float32),
              user_features=array([[3.3333333, 3.7272727, 3.7876713, ..., 3.7619047, 3.6666667,
        3.375    ],
       [3.3333333, 4.631579 , 4.8235292, ..., 4.263158 , 4.25     ,
        4.5      ],
       [3.3333333, 4.114754 , 4.22     , ..., 4.26     , 4.6666665,
        4.3333335],
       ...,
       [3.3333333, 3.25     , 3.66     , ..., 4.0454545, 4.       ,
        3.5704882],
       [3.3333333, 3.5      , 4.       , ..., 4.       , 4.6666665,
        3.       ],
       [3.3333333, 3.3333333, 3.3113208, ..., 3.440678 , 3.7857144,
        3.4166667]], dtype=float32))

In [126]:
print_RMSE(model,X_train,X_test,y_train,y_test)

Train RMSE: 0.8793016362064804
Test RMSE: 0.8761539215945888
